# 📊 포트폴리오 종합 현황 (01~05 통합)
자산 현황 · 인출 계획 · 시뮬레이션 · 위험 점수 · 리밸런싱을 한 화면에서 확인합니다.
> **Restart & Run All** 로 최신 데이터를 반영하세요.

In [ ]:
# ════════════════════════════════════════════════
#  공통 데이터 로드 — 모든 섹션이 여기서 시작
# ════════════════════════════════════════════════
import pandas as pd
import numpy as np
import yaml, sqlite3
import plotly.graph_objects as go
import plotly.offline
from plotly.subplots import make_subplots
from IPython.display import display, HTML
from pathlib import Path
from datetime import date

PROJECT_ROOT = Path.cwd()
TODAY        = date.today()
TODAY_STR    = TODAY.strftime('%Y년 %m월 %d일')
MONTH_STR    = TODAY.strftime('%Y-%m')

# ── config ──────────────────────────────────────
with open(PROJECT_ROOT / 'config.yaml', encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

MONTHLY_EXPENSE = CONFIG['user']['monthly_expense']
TARGET          = CONFIG['portfolio']
INFLATION       = CONFIG.get('inflation', {}).get('assumed_rate', 0.025)
BUCKET_MAP      = {'cash':1,'bond':2,'tdf':2,'fund':2,'equity':3,'income':3}

PENSION_CFG   = CONFIG.get('income',{}).get('national_pension',{})
base_pension  = PENSION_CFG.get('base_amount', 0)
pension_start = PENSION_CFG.get('start_date', None)

if pension_start:
    py, pm = map(int, pension_start.split('-'))
    if TODAY >= date(py, pm, 1):
        yrs = (TODAY.year-py) + (TODAY.month-pm)/12
        pension_income = base_pension * (1+INFLATION)**yrs
        months_to_pension = 0
    else:
        pension_income = 0
        months_to_pension = (py-TODAY.year)*12 + (pm-TODAY.month)
else:
    pension_income = 0
    months_to_pension = None

net_withdrawal = max(0, MONTHLY_EXPENSE - pension_income)

# ── 자산 ────────────────────────────────────────
df = pd.read_csv(PROJECT_ROOT/'assets.csv', encoding='utf-8-sig')
mask = df['current_value'].isna() | (df['current_value']==0)
df.loc[mask,'current_value'] = df.loc[mask,'quantity'] * df.loc[mask,'unit_price']
df['current_value'] = pd.to_numeric(df['current_value'], errors='coerce').fillna(0)
if 'is_active' not in df.columns: df['is_active'] = 1
df = df[df['is_active'].fillna(1).astype(int)==1].reset_index(drop=True)
df['bucket'] = df['asset_type'].str.lower().map(BUCKET_MAP).fillna(0).astype(int)

bucket_sum = df.groupby('bucket')['current_value'].sum()
type_sum   = df.groupby('asset_type')['current_value'].sum()
total      = df['current_value'].sum()
b1 = bucket_sum.get(1,0); b2 = bucket_sum.get(2,0); b3 = bucket_sum.get(3,0)

# ── DB ──────────────────────────────────────────
db_path = PROJECT_ROOT / 'portfolio.db'
conn = sqlite3.connect(db_path)

wd_df = pd.read_sql_query(
    "SELECT amount, actual_amount, guardrail_applied, note FROM withdrawal_log ORDER BY date DESC LIMIT 1", conn)
if not wd_df.empty:
    recommended_wd = wd_df.iloc[0]['amount'] or 0
    actual_wd      = wd_df.iloc[0]['actual_amount']
    display_wd     = actual_wd if actual_wd is not None else recommended_wd
    wd_label       = '실제 인출액' if actual_wd is not None else '권장 인출액'
else:
    recommended_wd = net_withdrawal; actual_wd = None
    display_wd = net_withdrawal; wd_label = '권장 인출액'

risk_df = pd.read_sql_query(
    "SELECT total_score,cash_score,seq_score,conc_score,level FROM risk_scores ORDER BY date DESC LIMIT 1", conn)
conn.close()

if not risk_df.empty:
    r = risk_df.iloc[0]
    total_score = r['total_score']; cash_score = r['cash_score']
    seq_score   = r['seq_score'];   conc_score  = r['conc_score']
    risk_level  = r['level']
else:
    # 즉석 계산
    months_covered = b1/MONTHLY_EXPENSE if MONTHLY_EXPENSE>0 else 0
    cash_score = 0 if months_covered>=12 else 30 if months_covered>=6 else 60 if months_covered>=3 else 100
    eq_ratio = (type_sum.get('equity',0)+type_sum.get('income',0))/total if total>0 else 0
    seq_score = 20 if eq_ratio<=0.35 else 50 if eq_ratio<=0.50 else 70
    cur = {'cash':type_sum.get('cash',0)/total,'bond':(type_sum.get('bond',0)+type_sum.get('tdf',0)+type_sum.get('fund',0))/total,
           'equity':type_sum.get('equity',0)/total,'income':type_sum.get('income',0)/total}
    tgt = {'cash':TARGET['target_cash'],'bond':TARGET['target_bond'],'equity':TARGET['target_equity'],'income':TARGET['target_income']}
    max_dev = max(abs(cur[k]-tgt[k]) for k in tgt)
    conc_score = 100 if max_dev>=0.20 else 50 if max_dev>=0.10 else 0
    total_score = cash_score*0.4 + seq_score*0.4 + conc_score*0.2
    risk_level  = 'green' if total_score<=25 else 'yellow' if total_score<=55 else 'red'

risk_label_map = {'green':'🟢 녹색 (안전)','yellow':'🟡 황색 (주의)','red':'🔴 적색 (위험)'}
risk_label = risk_label_map.get(risk_level,'🟡 황색 (주의)')

print(f'✅ 데이터 로드 완료 — {TODAY_STR}')
print(f'   총 자산: {total:,.0f}원  |  활성 자산: {len(df)}개')
print(f'   이번 달 {wd_label}: {display_wd:,.0f}원')
print(f'   위험 등급: {risk_label}')
# plotly.js 한 번만 로드
display(HTML('<script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>'))


In [ ]:
# ════════════════════════════════════════════════
#  ① 포트폴리오 핵심 지표 & 버킷 배분
# ════════════════════════════════════════════════
annual_rate = (display_wd*12)/total*100 if total>0 else 0
months_covered_b1 = b1/MONTHLY_EXPENSE if MONTHLY_EXPENSE>0 else 0

# ── 상단 요약 카드 ──────────────────────────────
card_style = 'background:{bg};border-radius:10px;padding:14px 18px;text-align:center;'
def card(bg, icon, label, value, sub=''):
    return (
        f'<div style="{card_style.format(bg=bg)}">'
        f'<div style="font-size:22px">{icon}</div>'
        f'<div style="font-size:12px;color:#555;margin:2px 0">{label}</div>'
        f'<div style="font-size:18px;font-weight:bold;color:#2c3e50">{value}</div>'
        f'{"<div style=\'font-size:11px;color:#888\'>"+sub+"</div>" if sub else ""}'
        f'</div>'
    )

cards_html = (
    '<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:10px;margin-bottom:20px">'
    + card('#eaf4fb','💰','총 자산',f'{total/1e8:.2f}억원',f'{total:,.0f}원')
    + card('#e8f8f0','🏦','버킷1 현금성',f'{b1/1e8:.2f}억원',f'{b1/total*100:.1f}%  ({months_covered_b1:.0f}개월치)')
    + card('#fef9e7','📈','버킷3 성장',f'{b3/1e8:.2f}억원',f'{b3/total*100:.1f}%')
    + card('#f5eef8','💸',wd_label,f'{display_wd/1e4:.0f}만원/월',f'연 인출률 {annual_rate:.1f}%')
    + '</div>'
)
display(HTML(f'<h3 style="color:#2c3e50;margin-bottom:8px">① 포트폴리오 현황 — {TODAY_STR}</h3>'))
display(HTML(cards_html))

# ── 버킷 배분 파이 + 자산유형 도넛 차트 ──────────
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type':'pie'},{'type':'pie'}]],
    subplot_titles=['버킷별 배분','자산 유형별 배분']
)

# 버킷 파이
bucket_labels = ['버킷1 현금성','버킷2 채권/펀드','버킷3 성장/인컴']
bucket_vals   = [b1, b2, b3]
bucket_colors = ['#3498db','#9b59b6','#e67e22']
fig.add_trace(go.Pie(
    labels=bucket_labels, values=bucket_vals,
    marker_colors=bucket_colors,
    textinfo='label+percent', hole=0.35,
    textfont_size=12
), row=1, col=1)

# 자산유형 도넛
type_order  = ['cash','bond','fund','tdf','equity','income']
type_labels = {'cash':'현금성예금','bond':'채권','fund':'펀드','tdf':'TDF','equity':'주식','income':'리츠/인컴'}
type_colors = {'cash':'#3498db','bond':'#8e44ad','fund':'#16a085','tdf':'#1abc9c','equity':'#e67e22','income':'#27ae60'}
t_vals = []; t_lbls = []; t_cols = []
for t in type_order:
    v = type_sum.get(t, 0)
    if v > 0:
        t_vals.append(v); t_lbls.append(type_labels[t]); t_cols.append(type_colors[t])

fig.add_trace(go.Pie(
    labels=t_lbls, values=t_vals,
    marker_colors=t_cols,
    textinfo='label+percent', hole=0.45,
    textfont_size=11
), row=1, col=2)

fig.update_layout(
    height=380, margin=dict(t=40,b=20,l=20,r=20),
    showlegend=True, legend=dict(orientation='h', y=-0.05)
)
display(HTML(plotly.offline.plot(fig, output_type='div', include_plotlyjs=False)))

# ── 계좌별 자산 bar ─────────────────────────────
acct = df.groupby('account_name')['current_value'].sum().sort_values(ascending=True)
fig2 = go.Figure(go.Bar(
    y=acct.index, x=acct.values,
    orientation='h',
    marker_color='#3498db',
    text=[f'{v/1e8:.2f}억' for v in acct.values],
    textposition='outside'
))
fig2.update_layout(
    title='계좌별 자산 규모', height=200,
    margin=dict(t=40,b=20,l=120,r=80),
    xaxis=dict(showticklabels=False)
)
display(HTML(plotly.offline.plot(fig2, output_type='div', include_plotlyjs=False)))

In [ ]:
# ════════════════════════════════════════════════
#  ② 이번 달 인출 계획 & 포트폴리오 지속 시뮬레이션
# ════════════════════════════════════════════════
display(HTML('<h3 style="color:#2c3e50;margin:20px 0 8px">② 인출 계획 & 지속 가능 시뮬레이션</h3>'))

# ── 인출 정보 카드 ──────────────────────────────
pension_str = (
    f'{pension_income:,.0f}원/월 수령 중' if pension_income>0
    else f'개시까지 {months_to_pension}개월 ({pension_start} 예정)' if pension_start
    else '해당 없음'
)
info_html = (
    '<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:10px;margin-bottom:16px">'
    + card('#fef9e7','🏠','월 생활비',f'{MONTHLY_EXPENSE/1e4:.0f}만원','고정 지출')
    + card('#e8f8f0','🎫','국민연금',pension_str if pension_income==0 else f'{pension_income/1e4:.0f}만원/월',pension_str if pension_income>0 else '')
    + card('#fdebd0','💳',wd_label,f'{display_wd/1e4:.0f}만원/월',f'권장: {recommended_wd/1e4:.0f}만원' if actual_wd is not None else '실제 인출액 미입력')
    + '</div>'
)
display(HTML(info_html))

# ── 30년 시뮬레이션 ──────────────────────────────
SIM_YEARS = 35
b1_r, b2_r, b3_r = 0.015, 0.030, 0.060

if pension_start:
    py2, pm2 = map(int, pension_start.split('-'))
    years_to_pension = (py2-TODAY.year) + (pm2-TODAY.month)/12
else:
    years_to_pension = 999

sim_b1, sim_b2, sim_b3 = float(b1), float(b2), float(b3)
years_x, total_y = [0], [sim_b1+sim_b2+sim_b3]
b1_y, b2_y, b3_y  = [sim_b1],[sim_b2],[sim_b3]
runout_year = None

for yr in range(1, SIM_YEARS+1):
    sim_expense = MONTHLY_EXPENSE * 12 * (1+INFLATION)**yr
    if base_pension>0 and yr>=years_to_pension:
        yrs_since_p = yr - years_to_pension
        annual_pension = base_pension*12*(1+INFLATION)**yrs_since_p
    else:
        annual_pension = 0
    net_annual = max(0, sim_expense - annual_pension)

    sim_b1 -= net_annual
    sim_b1 *= (1+b1_r); sim_b2 *= (1+b2_r); sim_b3 *= (1+b3_r)

    # 버킷 1 부족 시 버킷 2에서 보충
    if sim_b1 < MONTHLY_EXPENSE*6:
        refill = min(sim_b2, MONTHLY_EXPENSE*12 - sim_b1)
        sim_b1 += refill; sim_b2 -= refill
    # 버킷 2 부족 시 버킷 3에서 보충
    if sim_b2 < MONTHLY_EXPENSE*12:
        refill = min(sim_b3, MONTHLY_EXPENSE*24 - sim_b2)
        sim_b2 += refill; sim_b3 -= refill

    sim_b1 = max(sim_b1, 0); sim_b2 = max(sim_b2, 0); sim_b3 = max(sim_b3, 0)
    t = sim_b1+sim_b2+sim_b3
    if t<=0 and runout_year is None: runout_year = yr

    years_x.append(yr); total_y.append(t)
    b1_y.append(sim_b1); b2_y.append(sim_b2); b3_y.append(sim_b3)

year_labels = [str(TODAY.year+y) for y in years_x]

fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=year_labels, y=[v/1e8 for v in b3_y], name='버킷3 성장',
    fill='tozeroy', line=dict(color='#e67e22',width=2), stackgroup='one'))
fig3.add_trace(go.Scatter(x=year_labels, y=[v/1e8 for v in b2_y], name='버킷2 채권',
    fill='tonexty', line=dict(color='#9b59b6',width=2), stackgroup='one'))
fig3.add_trace(go.Scatter(x=year_labels, y=[v/1e8 for v in b1_y], name='버킷1 현금',
    fill='tonexty', line=dict(color='#3498db',width=2), stackgroup='one'))

# 연금 개시 수직선 (Scatter로 그려야 categorical x축에서 안정적)
if pension_start and 0 < years_to_pension < SIM_YEARS:
    p_idx = max(0, min(int(round(years_to_pension)), len(year_labels)-1))
    p_x   = year_labels[p_idx]
    p_max = max(total_y) / 1e8 * 1.15
    fig3.add_trace(go.Scatter(
        x=[p_x, p_x], y=[0, p_max],
        mode='lines', line=dict(color='#27ae60', dash='dash', width=2),
        name=f'연금 개시 ({pension_start})', showlegend=True
    ))

title_str = f'포트폴리오 35년 지속 시뮬레이션'
if runout_year:
    title_str += f'  ⚠️ {TODAY.year+runout_year}년 소진 예상'
else:
    title_str += f'  ✅ {TODAY.year+SIM_YEARS}년까지 유지'

fig3.update_layout(
    title=title_str, height=380,
    yaxis_title='자산 규모 (억원)',
    legend=dict(orientation='h', y=1.08),
    margin=dict(t=60,b=40,l=60,r=20),
    hovermode='x unified'
)
display(HTML(plotly.offline.plot(fig3, output_type='div', include_plotlyjs=False)))

In [ ]:
# ════════════════════════════════════════════════
#  ③ 위험 점수 분석
# ════════════════════════════════════════════════
display(HTML('<h3 style="color:#2c3e50;margin:20px 0 8px">③ 위험 점수 분석</h3>'))

needle_color = {'green':'#00B050','yellow':'#FFC000','red':'#FF0000'}.get(risk_level,'#FFC000')

fig4 = make_subplots(
    rows=1, cols=2,
    specs=[[{'type':'indicator'},{'type':'bar'}]],
    subplot_titles=[f'종합 위험 게이지', '항목별 위험 점수']
)
fig4.add_trace(go.Indicator(
    mode='gauge+number',
    value=total_score,
    title={'text': risk_label, 'font':{'size':14}},
    number={'suffix':'점'},
    gauge={
        'axis': {'range':[0,100]},
        'bar':  {'color': needle_color},
        'steps':[
            {'range':[0,25],  'color':'#E2EFDA'},
            {'range':[25,55], 'color':'#FFF2CC'},
            {'range':[55,100],'color':'#FCE4D6'},
        ],
        'threshold':{'line':{'color':'black','width':3},'thickness':0.75,'value':total_score}
    }
), row=1, col=1)

cats   = ['현금 위험','순서 위험','집중 위험']
scores = [cash_score, seq_score, conc_score]
bcolors = ['#00B050' if s<=25 else '#FFC000' if s<=55 else '#FF0000' for s in scores]
fig4.add_trace(go.Bar(
    x=cats, y=scores, marker_color=bcolors,
    text=[f'{s:.0f}점' for s in scores], textposition='outside'
), row=1, col=2)
fig4.update_yaxes(range=[0,115], row=1, col=2)
fig4.update_layout(height=360, margin=dict(t=40,b=20,l=20,r=20), showlegend=False)
display(HTML(plotly.offline.plot(fig4, output_type='div', include_plotlyjs=False)))

# 위험 등급 설명
action_map = {
    'green':  '✅ 현재 포트폴리오를 유지하세요.',
    'yellow': '⚠️ 1~2개월 내 포트폴리오 점검을 권장합니다.',
    'red':    '🚨 즉시 리밸런싱 또는 버킷 충전이 필요합니다.'
}
display(HTML(
    f'<div style="background:#f8f9fa;border-left:4px solid {needle_color};'
    f'padding:10px 14px;border-radius:4px;margin-top:6px;font-size:13px">'
    f'<b>종합 {total_score:.1f}점 / 100점</b> &nbsp;│&nbsp; {risk_label}<br>'
    f'{action_map.get(risk_level,"")}</div>'
))

In [ ]:
# ════════════════════════════════════════════════
#  ④ 현재 vs 목표 배분 & 리밸런싱 액션
# ════════════════════════════════════════════════
display(HTML('<h3 style="color:#2c3e50;margin:20px 0 8px">④ 자산 배분 & 리밸런싱</h3>'))

labels_map = {'cash':'현금성','bond':'채권/펀드','equity':'주식형','income':'리츠/인컴'}
current_amt = {
    'cash':   type_sum.get('cash', 0),
    'bond':   type_sum.get('bond',0)+type_sum.get('tdf',0)+type_sum.get('fund',0),
    'equity': type_sum.get('equity', 0),
    'income': type_sum.get('income', 0),
}
target_ratio = {
    'cash':   TARGET['target_cash'],
    'bond':   TARGET['target_bond'],
    'equity': TARGET['target_equity'],
    'income': TARGET['target_income'],
}
threshold = CONFIG.get('portfolio',{}).get('rebalance_threshold', 0.10)

cats_r    = list(labels_map.values())
cur_pct   = [current_amt[k]/total*100 if total>0 else 0 for k in labels_map]
tgt_pct   = [target_ratio[k]*100 for k in labels_map]
diff_amt  = {k: target_ratio[k]*total - current_amt[k] for k in labels_map}

fig5 = make_subplots(rows=1, cols=2,
    subplot_titles=['현재 vs 목표 배분 (%)', '리밸런싱 필요 금액 (만원)'])

# 현재 vs 목표 bar
fig5.add_trace(go.Bar(name='현재', x=cats_r, y=cur_pct,
    marker_color='#3498db', text=[f'{v:.1f}%' for v in cur_pct], textposition='outside'), row=1, col=1)
fig5.add_trace(go.Bar(name='목표', x=cats_r, y=tgt_pct,
    marker_color='#bdc3c7', text=[f'{v:.0f}%' for v in tgt_pct], textposition='outside'), row=1, col=1)

# 리밸런싱 차이 bar
diff_vals  = [diff_amt[k]/1e4 for k in labels_map]
diff_colors = ['#27ae60' if v>=0 else '#e74c3c' for v in diff_vals]
fig5.add_trace(go.Bar(x=cats_r, y=diff_vals, marker_color=diff_colors,
    text=[f'{"+" if v>=0 else ""}{v:,.0f}만' for v in diff_vals],
    textposition='outside', name='조정액'), row=1, col=2)
fig5.add_hline(y=0, line_color='gray', row=1, col=2)

fig5.update_layout(height=360, barmode='group',
    legend=dict(orientation='h',y=1.08),
    margin=dict(t=40,b=20,l=40,r=20))
fig5.update_yaxes(range=[0, max(max(cur_pct),max(tgt_pct))*1.25], row=1, col=1)
fig5.update_yaxes(range=[min(min(diff_vals)*1.3,-5), max(max(diff_vals)*1.3,5)], row=1, col=2)
display(HTML(plotly.offline.plot(fig5, output_type='div', include_plotlyjs=False)))

# 리밸런싱 액션 테이블
needs_rebal = any(abs(current_amt[k]/total - target_ratio[k]) >= threshold for k in labels_map if total>0)
action_rows = ''
for k, label in labels_map.items():
    cur_r = current_amt[k]/total if total>0 else 0
    tgt_r = target_ratio[k]
    diff  = diff_amt[k]
    dev   = cur_r - tgt_r
    flag  = '⚠️ 리밸런싱 필요' if abs(dev)>=threshold else '정상'
    color = '#e74c3c' if abs(dev)>=threshold else '#27ae60'
    action = ('매수 필요' if diff>0 else '매도 필요') if abs(dev)>=threshold else '유지'
    action_rows += (
        f'<tr><td style="padding:6px 10px;font-weight:bold">{label}</td>'
        f'<td style="padding:6px 10px;text-align:right">{cur_r*100:.1f}%</td>'
        f'<td style="padding:6px 10px;text-align:right">{tgt_r*100:.0f}%</td>'
        f'<td style="padding:6px 10px;text-align:right;color:{color}">{dev*100:+.1f}%p</td>'
        f'<td style="padding:6px 10px;text-align:right">{diff/1e4:+,.0f}만원</td>'
        f'<td style="padding:6px 10px;color:{color};font-weight:bold">{flag}</td></tr>'
    )

status_txt = '⚠️ 리밸런싱 필요' if needs_rebal else '✅ 리밸런싱 불필요 (목표 배분 유지 중)'
status_color = '#e74c3c' if needs_rebal else '#27ae60'
display(HTML(
    f'<div style="font-weight:bold;color:{status_color};margin:10px 0 6px">{status_txt}</div>'
    '<div style="overflow-x:auto"><table style="border-collapse:collapse;width:100%;font-size:13px">'
    '<thead><tr style="background:#2c3e50;color:white">'
    '<th style="padding:7px 10px">자산군</th><th style="padding:7px 10px">현재</th>'
    '<th style="padding:7px 10px">목표</th><th style="padding:7px 10px">편차</th>'
    '<th style="padding:7px 10px">조정액</th><th style="padding:7px 10px">상태</th></tr></thead>'
    f'<tbody>{action_rows}</tbody></table></div>'
))

In [ ]:
# ════════════════════════════════════════════════
#  ⑤ 이달의 포트폴리오 요약
# ════════════════════════════════════════════════
display(HTML('<h3 style="color:#2c3e50;margin:20px 0 8px">⑤ 이달의 포트폴리오 요약</h3>'))

# DB에 저장된 AI 요약 조회 시도
summary_text = None
try:
    conn2 = sqlite3.connect(db_path)
    cur2  = conn2.cursor()
    cur2.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='summary_log'")
    if cur2.fetchone():
        row2 = pd.read_sql_query(
            "SELECT summary FROM summary_log ORDER BY date DESC LIMIT 1", conn2)
        if not row2.empty:
            summary_text = row2.iloc[0]['summary']
    conn2.close()
except:
    pass

if summary_text:
    display(HTML(
        '<div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;'
        'padding:16px;font-size:14px;line-height:1.7;white-space:pre-wrap">'
        + summary_text.replace('<','&lt;').replace('>','&gt;')
        + '</div>'
    ))
else:
    # AI 요약 미저장 시 자동 생성
    pension_line = (
        f'국민연금은 {pension_start}부터 월 {base_pension:,}원(물가 반영) 수령 예정이며, '
        f'개시까지 {months_to_pension}개월 남았습니다.' if months_to_pension and months_to_pension>0
        else f'국민연금을 현재 월 {pension_income:,.0f}원 수령 중입니다.' if pension_income>0
        else ''
    )
    rebal_line = (
        '현재 자산 배분이 목표 범위를 벗어났으므로 리밸런싱을 검토하시기 바랍니다.'
        if needs_rebal else '현재 자산 배분은 목표 범위 내에 있어 리밸런싱이 불필요합니다.'
    )
    sim_line = (
        f'시뮬레이션 결과 현재 인출 속도로는 {TODAY.year+runout_year}년경 자산이 소진될 수 있어 주의가 필요합니다.'
        if runout_year else
        f'시뮬레이션 결과 35년({TODAY.year+35}년)까지 자산이 유지될 것으로 예상됩니다.'
    )
    auto_summary = (
        f'[{TODAY_STR} 자동 요약]\n\n'
        f'총 자산은 {total/1e8:.2f}억원이며, 버킷1(현금성) {b1/1e8:.2f}억원·'
        f'버킷2(채권/펀드) {b2/1e8:.2f}억원·버킷3(성장) {b3/1e8:.2f}억원으로 구성되어 있습니다.\n\n'
        f'이번 달 {wd_label}은 {display_wd:,.0f}원(연 인출률 {annual_rate:.1f}%)이며, '
        f'{pension_line}\n\n'
        f'종합 위험 점수는 {total_score:.1f}점({risk_label})입니다. '
        f'{sim_line}\n\n'
        f'{rebal_line}\n\n'
        f'※ AI 상세 요약은 05_llm_summary.ipynb를 실행하세요.'
    )
    display(HTML(
        '<div style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:8px;'
        'padding:16px;font-size:14px;line-height:1.7;white-space:pre-wrap">'
        + auto_summary
        + '</div>'
    ))